In [8]:
!pip install youtube-comment-downloader

import pandas as pd
from itertools import islice
from youtube_comment_downloader import YoutubeCommentDownloader

url = "https://www.youtube.com/watch?v=ZvbifV4mKb0"
comments = YoutubeCommentDownloader().get_comments_from_url(url)

rows = [c["text"] for c in islice(comments, 100)]
df = pd.DataFrame({"comment": rows, "label": ""})
df = df.drop_duplicates(subset="comment")
df.to_csv("comments.csv", index=False)
print(df.shape)
df.head()

(99, 2)


,comment,label
0,It s agood game,
1,Very pathetic. It is sad to see Sony like this...,
2,"Meh, ....I give that sphere advertisement 6/10.",
3,Du physique du physique,
4,No physical games 📀📀 No money! ❌❌,


In [9]:
import pandas as pd

df = pd.read_csv("comments.csv")
df["label"] = df["label"].fillna("")
options = {"p": "Positive", "n": "Negative", "u": "Neutral"}

for i in range(len(df)):
    if df.loc[i, "label"] != "":
        continue
    print(f"\n[{i}] {df.loc[i, 'comment']}")
    key = input("p / n / u / x (skip) / s (stop): ").lower()
    if key == "s":
        break
    if key == "x":
        df.loc[i, "label"] = "SKIP"
        continue
    df.loc[i, "label"] = options.get(key, "Neutral")
    df.to_csv("comments_labeled.csv", index=False)

df.to_csv("comments_labeled.csv", index=False)
print(df["label"].value_counts())


[0] It s agood game
p / n / u / x (skip) / s (stop): p

[1] Very pathetic. It is sad to see Sony like this. When respect leaves, relationship follows!
p / n / u / x (skip) / s (stop): n

[2] Meh, ....I give that sphere advertisement 6/10.
p / n / u / x (skip) / s (stop): u

[3] Du physique du physique
p / n / u / x (skip) / s (stop): x

[4] No physical games 📀📀 No money! ❌❌
p / n / u / x (skip) / s (stop): n

[5] "play has no limits". If you follow the straight line that it does have limits.
p / n / u / x (skip) / s (stop): p

[6] So gta gets vinyl but games can’t get a disc? 
no PlayStation 
You ain’t getting away with this 
F Sony!
💿💿💿💿💿💿
p / n / u / x (skip) / s (stop): n

[7] No CDs  no DVDs  no BLU-RAYs no likes. no no no
p / n / u / x (skip) / s (stop): n

[8] Remember what disc do don't remove them
p / n / u / x (skip) / s (stop): n

[9] Bring back physical games !!!
p / n / u / x (skip) / s (stop): n

[10] We want granny game on ps5
p / n / u / x (skip) / s (stop): p

[11] 💿:


In [11]:
#let's build the model
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("comments_labeled.csv")
df = df[df["label"].isin(["Positive", "Negative", "Neutral"])]
print(df["label"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    df["comment"], df["label"],
    test_size=0.2, stratify=df["label"], random_state=42)

model = make_pipeline(
    TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2),
    LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

label
Negative    57
Neutral     20
Positive     6
Name: count, dtype: int64
Accuracy: 0.588
              precision    recall  f1-score   support

    Negative       0.69      0.75      0.72        12
     Neutral       0.25      0.25      0.25         4
    Positive       0.00      0.00      0.00         1

    accuracy                           0.59        17
   macro avg       0.31      0.33      0.32        17
weighted avg       0.55      0.59      0.57        17

[[9 3 0]
 [3 1 0]
 [1 0 0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
